In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
print(f"Current path: {Path.cwd()}")

Current path: /app


In [3]:
from icare_risk.utils import convert_csvs_to_parquet
path = './data/synthetic/2026-09-04_234319'
convert_csvs_to_parquet(directory_path=path, delete_originals=False)

Converting: icare_pharmacy_prescribing_anon.csv -> icare_pharmacy_prescribing_anon.parquet
Converting: icare_problems_anon.csv -> icare_problems_anon.parquet
Converting: icare_microbiology_anon.csv -> icare_microbiology_anon.parquet
Converting: icare_episodes_anon.csv -> icare_episodes_anon.parquet
Converting: icare_vital_signs_anon.csv -> icare_vital_signs_anon.parquet
Converting: icare_pathology_blood_anon.csv -> icare_pathology_blood_anon.parquet
Successfully converted 6 files.


In [4]:
import pandas as pd
from icare_risk.core.dataset import ClinicalDataset
from icare_risk.phenotypes.sample import compute_fever

In [4]:
def build_feature_matrix(episodes_path: str, table_paths: dict) -> pd.DataFrame:
    # 1. Initialize the dataset engine
    dataset = ClinicalDataset(episodes_path, table_paths)
    
    # 2. Compute requested phenotypes
    fever_df = compute_fever(dataset)
    
    # 3. Build the final matrix
    # Assuming episodes.parquet is the base table we want to enrich
    base_df = pd.read_parquet(episodes_path)[['subject', 'admission_time']]
    
    # Merge features
    final_df = pd.merge(base_df, fever_df, on=['subject', 'admission_time'], how='left')
    final_df = pd.merge(final_df, esbl_df, on=['subject', 'admission_time'], how='left')
    
    # Fill missing with False (meaning no evidence of phenotype)
    final_df.fillna(False, inplace=True)
    
    return final_df

In [4]:
from icare_risk.core.dataset import ClinicalDataset

if __name__ == "__main__":

    # Paths to your downloaded Snowflake data
    episodes_path = f'{path}/icare_episodes_anon.parquet'
    table_paths = {
        'vitals': f'{path}/icare_vital_signs_anon.parquet',
        'pathology': f'{path}/icare_pathology_blood_anon.parquet',
        'microbiology': f'{path}/icare_microbiology_anon.parquet',
        'problems': f'{path}/icare_problems_anon.parquet'
    }

    # Map the unique column names from the episodes table
    episode_config = {
        'admission_date': 'ADMISSION_DATE',
        'admission_time': 'ADMISSION_TIME',
        'discharge_date': 'DISCHARGE_DATE'
    }

    # Map the most clinically relevant timestamp for each table
    table_config = {
        'vitals':       {'time_column': 'OBSERVATION_PERFORMED_DT'},
        'pathology':    {'time_column': 'SAMPLE_COLLECTED_DT'},
        'microbiology': {'time_column': 'LATEST_COLLECT_DT'},
        'problems':     {'time_column': 'PROBLEM_DT_TM'}
    }

    # Initialize the engine
    dataset = ClinicalDataset(
        episodes_path=episodes_path, 
        episode_config=episode_config, 
        table_paths=table_paths, 
        table_config=table_config
    )

In [5]:
def compute_hypoxia(dataset: ClinicalDataset):
    # Safely fetches ONLY vitals during the current admission
    vitals = dataset.get_current_stay('vitals')
    
    # Filter to Oxygen Saturation rows AND explicitly create a copy
    o2_mask = vitals['OBSERVATION_NAME'] == 'Oxygen Saturation'
    o2_vitals = vitals[o2_mask].copy()
    
    # Logic: is Oxygen < 90% at any point during this stay?
    # This will now execute perfectly without warnings
    o2_vitals['is_hypoxic'] = o2_vitals['OBSERVATION_RESULT_CLEAN'] < 90.0
    
    # Group by the standardized columns
    results = o2_vitals.groupby(['SUBJECT', 'std_admission_time'])['is_hypoxic'].any()
    
    return results.reset_index()

df_hypoxia = compute_hypoxia(dataset)
print(df_hypoxia)

    SUBJECT  std_admission_time  is_hypoxic
0     10001 2022-04-13 02:24:00        True
1     10002 2023-08-17 14:13:00        True
2     10003 2020-07-30 13:03:00        True
3     10004 2022-11-18 16:48:00        True
4     10005 2022-10-25 03:50:00        True
..      ...                 ...         ...
95    10096 2020-06-19 00:35:00        True
96    10097 2023-08-24 09:05:00        True
97    10098 2021-12-30 05:02:00        True
98    10099 2021-09-06 08:51:00        True
99    10100 2020-04-05 01:08:00        True

[100 rows x 3 columns]


In [6]:
def compute_test_phenotype(dataset: ClinicalDataset):
    vitals = dataset.get_current_stay('vitals')
    pathology = dataset.get_current_stay('pathology')
    problems = dataset.get_current_stay('problems')

    print(vitals.shape, vitals.columns.tolist())
    print(pathology.shape, pathology.columns.tolist())
    print(vitals.SUBJECT.nunique())

    hx_vitals = dataset.get_historical('vitals')
    v2 = dataset.get_current_stay('vitals', time_window_hours=2)
    print(vitals.shape)
    print(hx_vitals.shape)
    print(v2.shape)

compute_test_phenotype(dataset)


(27760, 11) ['SUBJECT', 'ENCNTR_ID', 'OBSERVATION_CODE', 'OBSERVATION_NAME', 'OBSERVATION_PERFORMED_DT', 'OBSERVATION_START_DT', 'OBSERVATION_END_DT', 'OBSERVATION_RESULT_CLEAN', 'OBSERVATION_UNIT', 'standard_time', 'std_admission_time']
(3827, 15) ['SUBJECT', 'PBAID', 'LABORATORY_DEPARTMENT', 'ORDER_CODE', 'ORDER_NAME', 'RESULT_CLEANED', 'RESULT_LOWER_RANGE', 'RESULT_UPPER_RANGE', 'SAMPLE_COLLECTED_DT', 'RESULT_AVAILABLE_DT', 'TEST_CODE', 'TEST_NAME', 'TEST_RESULT_UNIT', 'standard_time', 'std_admission_time']
100
(27760, 11)
(2107, 11)
(265, 11)


In [7]:
from icare_risk.phenotypes.sample import compute_fever

df_fever = compute_fever(dataset)
print(df_fever.shape, df_fever.fever_phenotype.sum())

(100, 3) 99


In [25]:
import pandas as pd
from typing import List

def print_patient_timeline(dataset, subject_id: int, table_name: str = 'vitals') -> None:
    """
    Visualizes the temporal boundaries of a single clinical table for a specific patient.
    
    This function queries the patient's hospital stays and provides a compact 
    summary of event counts and edge timestamps (First/Last) for both historical 
    and current events relative to each admission.

    Args:
        dataset (ClinicalDataset): The initialized data provider engine.
        subject_id (int): The unique identifier (SUBJECT) of the patient.
        table_name (str): The name of the clinical table to visualize (e.g., 'vitals').

    Returns:
        None. Prints the timeline directly to the console.
    """
    print(f"=== TIMELINE FOR PATIENT {subject_id} | TABLE: {table_name.upper()} ===")
    
    # 1. Fetch all hospital stays for this patient directly from the episodes view
    query = f"SELECT std_admission_time, std_discharge_time FROM episodes WHERE SUBJECT = {subject_id}"
    patient_stays = dataset.con.query(query).df()
    
    if patient_stays.empty:
        print("  Patient not found in episodes table.")
        return

    try:
        # 2. Fetch all current and historical data for this table
        curr = dataset.get_current_stay(table_name)
        hist = dataset.get_historical(table_name)
    except Exception as e:
        print(f"  Error fetching data for table '{table_name}': {e}")
        return
        
    # 3. Iterate through every known stay for this patient
    for _, stay in patient_stays.iterrows():
        adm = stay['std_admission_time']
        dis = stay['std_discharge_time']
        
        print(f"\nAdmission: {adm}  --->  Discharge: {dis}")
        
        # Filter the fetched data for THIS specific stay
        p_curr = curr[(curr['SUBJECT'] == subject_id) & (curr['std_admission_time'] == adm)]
        p_hist = hist[(hist['SUBJECT'] == subject_id) & (hist['std_admission_time'] == adm)]
        
        # Print Historical Summary
        if not p_hist.empty:
            print(f"  [HISTORICAL] {len(p_hist):<4} events. First: {p_hist['standard_time'].min()} | Last: {p_hist['standard_time'].max()}")
        else:
            print(f"  [HISTORICAL] No prior history found.")
            
        # Print Current Summary
        if not p_curr.empty:
            print(f"  [CURRENT]    {len(p_curr):<4} events. First: {p_curr['standard_time'].min()} | Last: {p_curr['standard_time'].max()}")
        else:
            print(f"  [CURRENT]    No events recorded during this specific stay.")
            
    print("\n" + "=" * 50)


def print_patient_multitable_timeline(dataset, subject_id: int, table_names: List[str]) -> None:
    """
    Succinctly visualizes the temporal boundaries across multiple clinical tables 
    for a specific patient's hospital stays.
    
    This function groups a patient's data by hospital admission. Under each admission, 
    it provides a compact summary of event counts and edge timestamps (First/Last) for 
    both historical and current events across every requested table. This is ideal for 
    quickly verifying cross-table data alignment and ensuring no data leakage occurred.

    Args:
        dataset (ClinicalDataset): The initialized data provider engine.
        subject_id (int): The unique identifier (SUBJECT) of the patient.
        table_names (List[str]): A list of table names to query (e.g., ['vitals', 'pathology', 'problems']).

    Returns:
        None. Prints the succinct multi-table timeline directly to the console.
    """
    print(f"=== MULTI-TABLE TIMELINE FOR PATIENT {subject_id} ===")
    
    # Find all unique hospital stays for this patient
    query = f"SELECT std_admission_time, std_discharge_time FROM episodes WHERE SUBJECT = {subject_id}"
    patient_stays = dataset.con.query(query).df()
    
    if patient_stays.empty:
        print("  Patient not found in episodes table.")
        return
        
    # Iterate through each unique hospital stay
    for _, stay in patient_stays.iterrows():
        adm = stay['std_admission_time']
        dis = stay['std_discharge_time']
        
        print(f"\n[STAY] {adm} to {dis}")
        
        # Query stats for each table requested
        for table in table_names:
            try:
                curr = dataset.get_current_stay(table)
                hist = dataset.get_historical(table)
                
                # Filter for this patient and this specific stay
                p_curr = curr[(curr['SUBJECT'] == subject_id) & (curr['std_admission_time'] == adm)]
                p_hist = hist[(hist['SUBJECT'] == subject_id) & (hist['std_admission_time'] == adm)]
                
                print(f"  > {table.upper()}:")
                
                # Print Historical Summary (Now showing First and Last)
                if not p_hist.empty:
                    print(f"      Historical: {len(p_hist):<4} records (First: {p_hist['standard_time'].min()} | Last: {p_hist['standard_time'].max()})")
                else:
                    print(f"      Historical: 0    records")
                    
                # Print Current Stay Summary (Showing First and Last)
                if not p_curr.empty:
                    print(f"      Current:    {len(p_curr):<4} records (First: {p_curr['standard_time'].min()} | Last: {p_curr['standard_time'].max()})")
                else:
                    print(f"      Current:    0    records")
                    
            except Exception as e:
                print(f"  > {table.upper()}: [Error fetching data: {str(e)}]")
                
    print("\n" + "=" * 50)


print_patient_multitable_timeline(dataset, 10001, 
    table_names=['vitals', 'pathology', 'microbiology', 'problems']
)
print_patient_timeline(dataset, 10001, table_name='vitals')

=== MULTI-TABLE TIMELINE FOR PATIENT 10001 ===

[STAY] 2022-04-13 02:24:00 to 2022-05-01 23:59:59
  > VITALS:
      Historical: 7    records (First: 2022-04-13 00:00:00 | Last: 2022-04-13 00:00:00)
      Current:    314  records (First: 2022-04-13 04:00:00 | Last: 2022-04-22 20:00:00)
  > PATHOLOGY:
      Historical: 0    records
      Current:    57   records (First: 2022-04-13 04:00:00 | Last: 2022-04-22 08:00:00)
  > MICROBIOLOGY:
      Historical: 0    records
      Current:    0    records
  > PROBLEMS:
      Historical: 2    records (First: 2020-01-23 00:00:00 | Last: 2022-01-04 00:00:00)
      Current:    0    records

=== TIMELINE FOR PATIENT 10001 | TABLE: VITALS ===

Admission: 2022-04-13 02:24:00  --->  Discharge: 2022-05-01 23:59:59
  [HISTORICAL] 7    events. First: 2022-04-13 00:00:00 | Last: 2022-04-13 00:00:00
  [CURRENT]    314  events. First: 2022-04-13 04:00:00 | Last: 2022-04-22 20:00:00



In [7]:
from icare_risk.phenotypes.pitt import compute_pitt_fever_score

df = compute_pitt_fever_score(dataset)
print(df)

['Oxygen Saturation' 'Body Temperature' 'Diastolic Blood Pressure'
 'Mean Arterial Pressure' 'Systolic Blood Pressure' 'Heart Rate'
 'Respiratory Rate']
    SUBJECT  std_admission_time  pitt_fever_score
0     10001 2022-04-13 02:24:00                 2
1     10002 2023-08-17 14:13:00                 2
2     10003 2020-07-30 13:03:00                 2
3     10004 2022-11-18 16:48:00                 2
4     10005 2022-10-25 03:50:00                 2
..      ...                 ...               ...
95    10096 2020-06-19 00:35:00                 2
96    10097 2023-08-24 09:05:00                 2
97    10098 2021-12-30 05:02:00                 2
98    10099 2021-09-06 08:51:00                 2
99    10100 2020-04-05 01:08:00                 2

[100 rows x 3 columns]


In [6]:
from icare_risk.phenotypes.pitt import compute_pitt_fever_score
from icare_risk.phenotypes.pitt import compute_pitt_hypo_score
df1 = compute_pitt_fever_score(dataset)
df2 = compute_pitt_hypo_score(dataset)
print(df1.head(2))
print(df2.head(2))

from icare_risk.phenotypes.sirs import derive_sirs_tachycardia
from icare_risk.phenotypes.sirs import derive_sirs_tachypnea
from icare_risk.phenotypes.sirs import derive_sirs_abnormal_temp
from icare_risk.phenotypes.sirs import derive_sirs_abnormal_wbc

df3 = derive_sirs_tachycardia(dataset)
df4 = derive_sirs_tachypnea(dataset)
df5 = derive_sirs_abnormal_temp(dataset)
df6 = derive_sirs_abnormal_wbc(dataset)
print(df3.head(2))
print(df4.head(2))
print(df5.head(2))
print(df6.head(2))

['Oxygen Saturation' 'Body Temperature' 'Diastolic Blood Pressure'
 'Mean Arterial Pressure' 'Systolic Blood Pressure' 'Heart Rate'
 'Respiratory Rate']
   SUBJECT  std_admission_time  pitt_fever_score
0    10001 2022-04-13 02:24:00                 2
1    10002 2023-08-17 14:13:00                 2
   SUBJECT  std_admission_time  pitt_hypo_score
0    10001 2022-04-13 02:24:00                2
1    10002 2023-08-17 14:13:00                2
   SUBJECT  std_admission_time  sirs_tachycardia_flag
0    10001 2022-04-13 02:24:00                      1
1    10002 2023-08-17 14:13:00                      1
   SUBJECT  std_admission_time  sirs_tachypnea_flag
0    10001 2022-04-13 02:24:00                    1
1    10002 2023-08-17 14:13:00                    1
   SUBJECT  std_admission_time  sirs_abnormal_temp_flag
0    10001 2022-04-13 02:24:00                        1
1    10002 2023-08-17 14:13:00                        1
   SUBJECT  std_admission_time  sirs_abnormal_wbc_flag
0    10001 2022

In [8]:
import pandas as pd

sirs_phenotypes_config = {
    # ---------------------------------------------------------
    # 1. SIRS Tachycardia (Heart Rate > 90 bpm)
    # ---------------------------------------------------------
    "sirs_tachycardia_flag": {
        "module": "icare_risk.phenotypes.utils",  # Update to icare_risk.engine if you moved it there
        "function": "derive_flag_from_yaml",
        "kwargs": {
            "output_name": "sirs_tachycardia_flag",
            "rules": [
                {
                    "table": "vitals",
                    "names": ["Heart Rate", "HR"],
                    "codes": [13472364],
                    "operator": ">",
                    "threshold": 90.0
                }
            ]
        }
    },

    # ---------------------------------------------------------
    # 2. SIRS Tachypnea (RR > 20 OR PaCO2 < 32 mmHg)
    # ---------------------------------------------------------
    "sirs_tachypnea_flag": {
        "module": "icare_risk.phenotypes.utils",
        "function": "derive_flag_from_yaml",
        "kwargs": {
            "output_name": "sirs_tachypnea_flag",
            "rules": [
                {
                    "table": "vitals",
                    "names": ["Respiratory Rate"],
                    "codes": [9096705],
                    "operator": ">",
                    "threshold": 20.0
                },
                {
                    "table": "pathology",
                    "names": ["PaCO2", "pCO2"],
                    "codes": [],
                    "operator": "<",
                    "threshold": 32.0
                }
            ]
        }
    },

    # ---------------------------------------------------------
    # 3. SIRS Abnormal Temperature (> 38.0°C OR < 36.0°C)
    # ---------------------------------------------------------
    "sirs_abnormal_temp_flag": {
        "module": "icare_risk.phenotypes.utils",
        "function": "derive_flag_from_yaml",
        "kwargs": {
            "output_name": "sirs_abnormal_temp_flag",
            "rules": [
                {
                    "table": "vitals",
                    "names": ["Temperature", "Body Temperature", "Temperature C"],
                    "codes": [10933766, 486347689],
                    "operator": ">",
                    "threshold": 38.0
                },
                {
                    "table": "vitals",
                    "names": ["Temperature", "Body Temperature", "Temperature C"],
                    "codes": [10933766, 486347689],
                    "operator": "<",
                    "threshold": 36.0
                }
            ]
        }
    },

    # ---------------------------------------------------------
    # 4. SIRS Abnormal WBC (> 12.0 OR < 4.0 OR Bands > 10%)
    # ---------------------------------------------------------
    "sirs_abnormal_wbc_flag": {
        "module": "icare_risk.phenotypes.utils",
        "function": "derive_flag_from_yaml",
        "kwargs": {
            "output_name": "sirs_abnormal_wbc_flag",
            "rules": [
                {
                    "table": "pathology",
                    "names": ["White Blood Cells", "WBC", "Leukocytes"],
                    "codes": ["wbc"],  # Added your 'wbc' string code from the docstring
                    "operator": ">",
                    "threshold": 12.0
                },
                {
                    "table": "pathology",
                    "names": ["White Blood Cells", "WBC", "Leukocytes"],
                    "codes": ["wbc"],
                    "operator": "<",
                    "threshold": 4.0
                },
                {
                    "table": "pathology",
                    "names": ["Bands", "Immature Granulocytes", "Band Neutrophils"],
                    "codes": ["bands"],
                    "operator": ">",
                    "threshold": 10.0
                }
            ]
        }
    }
}

class MockClinicalDataset:
    """
    A mock dataset designed to yield a mix of 0s and 1s across the SIRS criteria.
    """
    def get_current_stay(self, table_name):
        if table_name == 'vitals':
            return pd.DataFrame([
                # PATIENT 101: Classic Sepsis (Should score 4/4)
                [101, '2026-09-01', 'Heart Rate',       13472364, 115.0], # > 90 (Tachycardia: 1)
                [101, '2026-09-01', 'Respiratory Rate', 9096705,  26.0],  # > 20 (Tachypnea: 1)
                [101, '2026-09-01', 'Temperature C',    10933766, 39.2],  # > 38 (Abnormal Temp: 1)

                # PATIENT 102: Perfectly Healthy (Should score 0/4)
                [102, '2026-09-02', 'Heart Rate',       13472364, 75.0],  # Normal
                [102, '2026-09-02', 'Respiratory Rate', 9096705,  14.0],  # Normal
                [102, '2026-09-02', 'Temperature C',    10933766, 37.1],  # Normal

                # PATIENT 103: Hypothermic & Leukopenic (Temp: 1, WBC: 1)
                [103, '2026-09-03', 'Heart Rate',       13472364, 80.0],  # Normal
                [103, '2026-09-03', 'Respiratory Rate', 9096705,  18.0],  # Normal
                [103, '2026-09-03', 'Temperature C',    10933766, 35.1],  # < 36 (Abnormal Temp: 1)

                # PATIENT 104: Hyperventilation & Bandemia (Tachypnea: 1, WBC: 1)
                [104, '2026-09-04', 'Heart Rate',       13472364, 88.0],  # Normal
                [104, '2026-09-04', 'Respiratory Rate', 9096705,  18.0],  # Normal, but see PaCO2 below
                [104, '2026-09-04', 'Temperature C',    10933766, 37.5],  # Normal

                # PATIENT 105: Missing Labs (Tachycardia: 1, Temp: 1, WBC: 0)
                [105, '2026-09-05', 'Heart Rate',       13472364, 105.0], # > 90 (Tachycardia: 1)
                [105, '2026-09-05', 'Respiratory Rate', 9096705,  16.0],  # Normal
                [105, '2026-09-05', 'Temperature C',    10933766, 38.5]   # > 38 (Abnormal Temp: 1)
            ], columns=['SUBJECT', 'std_admission_time', 'OBSERVATION_NAME', 'OBSERVATION_CODE', 'OBSERVATION_RESULT_CLEAN'])

        elif table_name == 'pathology':
            return pd.DataFrame([
                # PATIENT 101: Classic Sepsis
                [101, '2026-09-01', 'White Blood Cells', 'wbc',   18.5],  # > 12 (Abnormal WBC: 1)
                [101, '2026-09-01', 'PaCO2',             None,    38.0],  # Normal
                
                # PATIENT 102: Perfectly Healthy
                [102, '2026-09-02', 'White Blood Cells', 'wbc',   7.0],   # Normal
                
                # PATIENT 103: Leukopenic
                [103, '2026-09-03', 'White Blood Cells', 'wbc',   2.5],   # < 4.0 (Abnormal WBC: 1)
                
                # PATIENT 104: Alternative Criteria triggers (PaCO2 and Bands)
                [104, '2026-09-04', 'White Blood Cells', 'wbc',   8.5],   # Normal WBC count
                [104, '2026-09-04', 'Bands',             'bands', 15.0],  # > 10% (Abnormal WBC: 1)
                [104, '2026-09-04', 'PaCO2',             None,    28.0]   # < 32 (Tachypnea: 1)
                
                # PATIENT 105 has no pathology records intentionally to test missing data handling
            ], columns=['SUBJECT', 'std_admission_time', 'ORDER_NAME', 'ORDER_CODE', 'RESULT_CLEANED'])
            
        return pd.DataFrame()

# ==========================================
# 4. THE PIPELINE RUNNER
# ==========================================
import importlib

def build_feature_matrix(dataset, config):
    """Iterates over the config, runs the functions, and joins the results."""
    feature_columns = []
    
    for feature_id, settings in config.items():
        print(f"Computing: {feature_id}...")
        
        # 1. Dynamically load the function based on the string in the config
        module_name = settings['module']
        function_name = settings['function']
        
        # If this was your real package, you'd use importlib:
        module = importlib.import_module(module_name)
        func = getattr(module, function_name)
        #func = globals()[function_name] # Using globals() here since everything is in one script
        
        # 2. Execute the function, passing the dataset and unpacking the kwargs
        feature_df = func(dataset=dataset, **settings['kwargs'])
        
        # 3. Set the index so pandas can easily join it with other features
        feature_df = feature_df.set_index(['SUBJECT', 'std_admission_time'])
        feature_columns.append(feature_df)
        
    # 4. Merge all individual flags into one wide dataframe
    final_matrix = pd.concat(feature_columns, axis=1)
    
    # Fill missing values with 0 (if a patient had no matching records, they get a 0 flag)
    return final_matrix.fillna(0).astype(int)

# --- Execute the Pipeline ---
dataset = MockClinicalDataset()
final_features = build_feature_matrix(dataset, sirs_phenotypes_config)

print("\n--- SUM ---")
print(final_features.sum(axis=0))
print("\n--- FINAL FEATURE MATRIX ---")
print(final_features)

Computing: sirs_tachycardia_flag...
Computing: sirs_tachypnea_flag...
Computing: sirs_abnormal_temp_flag...
Computing: sirs_abnormal_wbc_flag...

--- SUM ---
sirs_tachycardia_flag      2
sirs_tachypnea_flag        2
sirs_abnormal_temp_flag    3
sirs_abnormal_wbc_flag     3
dtype: int64

--- FINAL FEATURE MATRIX ---
                            sirs_tachycardia_flag  sirs_tachypnea_flag  \
SUBJECT std_admission_time                                               
101     2026-09-01                              1                    1   
102     2026-09-02                              0                    0   
103     2026-09-03                              0                    0   
104     2026-09-04                              0                    1   
105     2026-09-05                              1                    0   

                            sirs_abnormal_temp_flag  sirs_abnormal_wbc_flag  
SUBJECT std_admission_time                                                   
101     

In [22]:
def get_expected_sirs_output():
    """
    Returns the exact expected feature matrix for the MockClinicalDataset.
    """
    expected_df = pd.DataFrame([
        # [SUBJECT, std_admission_time, Tachycardia, Tachypnea, Temp, WBC, Total]
        [101, '2026-09-01', 1, 1, 1, 1, 4],  # Classic Sepsis
        [102, '2026-09-02', 0, 0, 0, 0, 0],  # Healthy
        [103, '2026-09-03', 0, 0, 1, 1, 2],  # Hypothermic & Leukopenic
        [104, '2026-09-04', 0, 1, 0, 1, 2],  # PaCO2 & Bands triggered
        [105, '2026-09-05', 1, 0, 1, 0, 2]   # Missing labs, defaults to 0
    ], columns=[
        'SUBJECT', 
        'std_admission_time', 
        'sirs_tachycardia_flag', 
        'sirs_tachypnea_flag', 
        'sirs_abnormal_temp_flag', 
        'sirs_abnormal_wbc_flag',
        'total_sirs_score'
    ])
    
    # Set the index to match the pipeline output
    expected_df = expected_df.set_index(['SUBJECT', 'std_admission_time'])
    
    return expected_df

# View the expected output
expected_output = get_expected_sirs_output()
print(expected_output)

                            sirs_tachycardia_flag  sirs_tachypnea_flag  \
SUBJECT std_admission_time                                               
101     2026-09-01                              1                    1   
102     2026-09-02                              0                    0   
103     2026-09-03                              0                    0   
104     2026-09-04                              0                    1   
105     2026-09-05                              1                    0   

                            sirs_abnormal_temp_flag  sirs_abnormal_wbc_flag  \
SUBJECT std_admission_time                                                    
101     2026-09-01                                1                       1   
102     2026-09-02                                0                       0   
103     2026-09-03                                1                       1   
104     2026-09-04                                0                       1   
105    

In [10]:
# -------------------------------------------------------------------------
# EXAMPLE 2. DERIVE HISTORICAL CONDITION
# -------------------------------------------------------------------------

# --------------
# Dataset
# --------------
class MockClinicalDataset:
    def get_history(self, table_name):
        if table_name == 'problems':
            # This is your exact CSV data, joined with a dummy std_admission_time
            # so the grouper knows which admission we are building features for.
            return pd.DataFrame([
                [10001, '2023-01-01', 'J44.9',  'COPD'],
                [10001, '2023-01-01', 'I10',    'Hypertension'],
                [10001, '2023-01-01', 'I25.21', 'Old Myocardial Infarction'], # Should match I25.2 prefix
                [10001, '2023-01-01', 'E11.9',  'Type 2 diabetes'],
                
                [10003, '2023-01-01', 'J44.9',  'COPD'],
                [10003, '2023-01-01', 'E11.9',  'Type 2 diabetes'],
                [10003, '2023-01-01', 'I10',    'Hypertension']
                # 10003 has no MI codes
            ], columns=['SUBJECT', 'std_admission_time', 'PROBLEM_CODE', 'PROBLEM_DESC'])
        return pd.DataFrame()

# --------------
# Configuration
# --------------
charlson_phenotypes_config = {
    "hx_mi_flag": {
        "module": "icare_risk.phenotypes.utils", # Use 'icare_risk.engine' in production
        "function": "derive_historical_condition",
        "kwargs": {
            "output_name": "hx_mi_flag",
            "icd10_codes": [],
            "snomed_codes": [],
            "icare_codes": ['57054005', '22298006', '1755008'],
            "target_codes": ['I21', 'I22', 'I25.2', '323..00', 'G30..00'],
            "sources": [
                {
                    "context_name": "problems",
                    "patient_col": "SUBJECT",
                    "code_col": "PROBLEM_CODE",
                    "vocab_key": "target_codes", # Tells it to use the target_codes list
                    "match_type": "prefix"
                }
            ]
        }
    },
    # Let's add COPD to show how easy it is to scale!
    "hx_copd_flag": {
        "module": "icare_risk.phenotypes.utils",
        "function": "derive_historical_condition",
        "kwargs": {
            "output_name": "hx_copd_flag",
            "target_codes": ['J44'], # J44 is COPD
            "sources": [
                {
                    "context_name": "problems",
                    "patient_col": "SUBJECT",
                    "code_col": "PROBLEM_CODE",
                    "vocab_key": "target_codes",
                    "match_type": "prefix"
                }
            ]
        }
    }
}

# ------------------------------
# Main
# ------------------------------
# --- Execute the Pipeline ---
dataset = MockClinicalDataset()
final_features = build_feature_matrix(dataset, charlson_phenotypes_config)

print("\n--- FINAL FEATURE MATRIX ---")
print(final_features)

Computing: hx_mi_flag...
Computing: hx_copd_flag...

--- FINAL FEATURE MATRIX ---
                            hx_mi_flag  hx_copd_flag
SUBJECT std_admission_time                          
10001   2023-01-01                   1             1
10003   2023-01-01                   0             1


In [ ]:
# ------------------------------------------------------------------------
# Increment ESBL
# ------------------------------------------------------------------------


increment_phenotypes_config = {
    "bsi_not_urinary_flag": {
        "module": "__main__",
        "function": "derive_bsi_not_urinary",
        "kwargs": {
            "output_name": "bsi_not_urinary_flag",
            "urine_culture_codes": ['LOINC-630-4']
        }
    },
    "bsi_non_ecoli_flag": {
        "module": "__main__",
        "function": "derive_bsi_non_ecoli",
        "kwargs": {
            "output_name": "bsi_non_ecoli_flag",
            "blood_culture_codes": ['LOINC-600-7']
        }
    }
}


dataset = MockClinicalDataset()
final_features = build_feature_matrix(dataset, increment_phenotypes_config)

print("\n--- FINAL FEATURE MATRIX ---")
print(final_features)

In [10]:
from icare_risk.phenotypes.pitt import compute_pitt_score
df_pitt = compute_pitt_score(dataset)
print(pitt)

['Oxygen Saturation' 'Body Temperature' 'Diastolic Blood Pressure'
 'Mean Arterial Pressure' 'Systolic Blood Pressure' 'Heart Rate'
 'Respiratory Rate']


CatalogException: Catalog Error: Table with name interventions does not exist!
Did you mean "pg_indexes"?